# EcoSort AI — dashboard and controlled retraining (Notebook 05)

**Purpose:** add the user-facing monitoring and data-update layer to the pipeline built in Notebooks 01–04.

This notebook creates:
- `dashboard.py`: Streamlit dashboard for API/model health, model information, visualizations, single-image prediction and bulk labelled-data upload.
- `src/retrain.py`: a controlled retraining job that uses the original training split plus newly labelled images, evaluates the candidate on the existing validation split, and promotes it only when the validation gate passes.
- `data_inbox/`: staging area for newly labelled images uploaded as a ZIP file.

**Important safeguards**
1. The held-out test split is not used for retraining, early stopping, candidate selection or promotion.
2. New uploads are added to the training data only after their class folders are validated.
3. A candidate is saved separately. The active metadata is updated only after the validation gate passes.
4. A successful upload or retraining run does not guarantee a better model. Inspect the recorded result.
5. The disposal mapping remains a separate, illustrative configuration; validate local disposal rules before operational use.

**Prerequisites:** run Notebooks 01–04 in the same EcoSort project folder. The expected Colab Drive path is `/content/drive/MyDrive/ecosort`.

## 1. Connect to the EcoSort project

In [1]:
from pathlib import Path
import os, sys, json, textwrap, subprocess, importlib.util

USE_DRIVE = True  # Set False only when the EcoSort project folder is the current directory.
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/ecosort")
else:
    PROJECT_DIR = Path.cwd()

MODELS_DIR = PROJECT_DIR / "models"
SPLIT_DIR = PROJECT_DIR / "splits"
REPORTS_DIR = PROJECT_DIR / "reports"
SRC_DIR = PROJECT_DIR / "src"
INBOX_DIR = PROJECT_DIR / "data_inbox"
for folder in [MODELS_DIR, SPLIT_DIR, REPORTS_DIR, SRC_DIR, INBOX_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

METADATA_PATH = MODELS_DIR / "model_metadata.json"
METRICS_PATH = REPORTS_DIR / "metrics.json"
assert METADATA_PATH.exists(), f"Missing {METADATA_PATH}. Run Notebook 03."
assert (SPLIT_DIR / "train.csv").exists() and (SPLIT_DIR / "val.csv").exists(), (
    "Missing train.csv or val.csv. Run Notebook 02."
)
metadata = json.loads(METADATA_PATH.read_text())
assert (MODELS_DIR / metadata["model_file"]).exists(), "Active model file is missing."
class_names = metadata["class_names"]
print("Project:", PROJECT_DIR)
print("Active model:", metadata["model_file"], "| version:", metadata["model_version"])
print("Classes:", class_names)
print("Staging folder:", INBOX_DIR)

Mounted at /content/drive
Project: /content/drive/MyDrive/ecosort
Active model: ecosort_mobilenetv2_v1.keras | version: v1
Classes: ['Cardboard', 'Food Organics', 'Glass', 'Metal', 'Miscellaneous Trash', 'Paper', 'Plastic', 'Textile Trash', 'Vegetation']
Staging folder: /content/drive/MyDrive/ecosort/data_inbox


## 2. Install dashboard dependencies

In [2]:
%pip -q install streamlit pandas matplotlib pillow requests

## 3. Add API uptime to the health endpoint

Notebook 04 already created `/health`, but it reports only whether the model is loaded. This small, explicit update adds the API process uptime in seconds. The API must be restarted after this change for the new field to appear.

In [3]:
API_PATH = SRC_DIR / "app.py"
assert API_PATH.exists(), "Missing src/app.py. Run Notebook 04 first."
api_text = API_PATH.read_text()

if "API_STARTED_AT" not in api_text:
    api_text = api_text.replace(
        "import os\n",
        "import os\nimport time\n",
        1
    )
    api_text = api_text.replace(
        "app = FastAPI(title=",
        "API_STARTED_AT = time.time()\n\napp = FastAPI(title=",
        1
    )
    api_text = api_text.replace(
        '"model_version": predictor.metadata["model_version"]\n',
        '"model_version": predictor.metadata["model_version"],\n'
        '        "uptime_seconds": round(time.time() - API_STARTED_AT, 2)\n',
        1
    )
    API_PATH.write_text(api_text)

print("Updated health endpoint:", API_PATH)
print("Restart the API process after running this notebook.")

Updated health endpoint: /content/drive/MyDrive/ecosort/src/app.py
Restart the API process after running this notebook.


## 4. Create the controlled retraining script

The retraining job starts from the currently active model and fine-tunes the MobileNetV2 feature extractor on:
- the original `train.csv` images, and
- any valid labelled images in `data_inbox/<class_name>/`.

It uses the original `val.csv` for early stopping and candidate comparison. It does **not** read `test.csv`.

**Promotion gate:** promote only when candidate validation loss is lower than the active model's validation loss and candidate validation accuracy is no more than 1 percentage point below the active model. Otherwise the active model stays unchanged. This is a conservative example gate, not a universal ML rule.

In [4]:
retrain_source = r"""
import json, os, shutil, datetime, tempfile
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf

ROOT = Path(os.getenv("ECOSORT_PROJECT_DIR", Path(__file__).resolve().parent.parent))
MODELS = ROOT / "models"
SPLITS = ROOT / "splits"
REPORTS = ROOT / "reports"
INBOX = ROOT / "data_inbox"
META_PATH = MODELS / "model_metadata.json"
METRICS_PATH = REPORTS / "metrics.json"
SEED = 42
BATCH = 32
IMG_SIZE = 224
EPOCHS = 5
LR = 1e-5
AUTOTUNE = tf.data.AUTOTUNE

def find_realwaste_root():
    raw = ROOT / "data_raw"
    for p in raw.rglob("RealWaste"):
        if p.is_dir() and any(x.is_dir() for x in p.iterdir()):
            return p
    raise FileNotFoundError("Could not locate data_raw/**/RealWaste class folders.")

def make_ds(frame, root, training=False):
    paths = [str(Path(root) / p) for p in frame["path"]]
    labels = frame["label"].astype("int32").to_numpy()
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    def load(path, label):
        raw = tf.io.read_file(path)
        image = tf.image.decode_image(raw, channels=3, expand_animations=False)
        image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
        image.set_shape((IMG_SIZE, IMG_SIZE, 3))
        return image, label
    ds = ds.map(load, num_parallel_calls=AUTOTUNE)
    if training:
        aug = tf.keras.Sequential([
            tf.keras.layers.RandomFlip("horizontal_and_vertical"),
            tf.keras.layers.RandomRotation(0.08),
            tf.keras.layers.RandomZoom(0.08),
            tf.keras.layers.RandomContrast(0.1),
        ])
        ds = ds.map(lambda x, y: (aug(x, training=True), y),
                    num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH).prefetch(AUTOTUNE)

def main():
    tf.keras.utils.set_random_seed(SEED)
    meta = json.loads(META_PATH.read_text())
    classes = meta["class_names"]
    active_path = MODELS / meta["model_file"]
    if not active_path.exists():
        raise FileNotFoundError(active_path)

    train = pd.read_csv(SPLITS / "train.csv")
    val = pd.read_csv(SPLITS / "val.csv")
    data_root = find_realwaste_root()

    # Add only staged, labelled images to the training frame.
    added = []
    allowed_ext = {".jpg", ".jpeg", ".png", ".bmp"}
    for class_name in classes:
        folder = INBOX / class_name
        if not folder.exists():
            continue
        for image_path in sorted(folder.iterdir()):
            if image_path.is_file() and image_path.suffix.lower() in allowed_ext:
                added.append({
                    "path": str(image_path.resolve()),
                    "label": classes.index(class_name)
                })

    if not added:
        raise RuntimeError(
            "No labelled images found in data_inbox/<class_name>/. "
            "Upload a ZIP containing one folder per class first."
        )

    # Original split paths are relative to the RealWaste image root;
    # staged upload paths are absolute and remain readable by Path(root) / path.
    extra = pd.DataFrame(added)
    combined = pd.concat([train[["path", "label"]], extra], ignore_index=True)
    # Recalculate balanced class weights after adding the labelled examples.
    counts = np.bincount(combined["label"].astype(int), minlength=len(classes))
    class_weights = {
        i: float(len(combined) / (len(classes) * counts[i]))
        for i in range(len(classes)) if counts[i] > 0
    }

    train_ds = make_ds(combined, data_root, training=True)
    val_ds = make_ds(val[["path", "label"]], data_root, training=False)

    active_model = tf.keras.models.load_model(active_path, compile=False)
    active_model.compile(
        optimizer=tf.keras.optimizers.Adam(LR),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    baseline_loss, baseline_acc = active_model.evaluate(val_ds, verbose=0)

    # Find the nested MobileNetV2 model and fine-tune its final layers only.
    nested_models = [layer for layer in active_model.layers
                     if isinstance(layer, tf.keras.Model)]
    if not nested_models:
        raise RuntimeError("Could not find the nested MobileNetV2 base model.")
    base = nested_models[0]
    base.trainable = True
    for layer in base.layers[:-30]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False

    active_model.compile(
        optimizer=tf.keras.optimizers.Adam(LR),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=2, restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=1, min_lr=1e-7
        )
    ]
    history = active_model.fit(
        train_ds, validation_data=val_ds, epochs=EPOCHS,
        class_weight=class_weights, callbacks=callbacks, verbose=1
    )
    candidate_loss, candidate_acc = active_model.evaluate(val_ds, verbose=0)

    # Keep candidate artifacts separate until the gate has passed.
    version_number = 1
    import re
    found = re.search(r"v(\d+)$", str(meta.get("model_version", "v1")))
    if found:
        version_number = int(found.group(1)) + 1
    candidate_version = f"v{version_number}"
    candidate_name = f"ecosort_mobilenetv2_{candidate_version}.keras"
    candidate_path = MODELS / candidate_name
    active_model.save(candidate_path)

    # Conservative, validation-only promotion rule.
    promoted = (
        candidate_loss < float(baseline_loss)
        and candidate_acc >= float(baseline_acc) - 0.01
    )

    result = {
        "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
        "previous_version": meta.get("model_version"),
        "candidate_version": candidate_version,
        "uploaded_images_used": len(added),
        "training_images_total": len(combined),
        "baseline_validation_loss": float(baseline_loss),
        "baseline_validation_accuracy": float(baseline_acc),
        "candidate_validation_loss": float(candidate_loss),
        "candidate_validation_accuracy": float(candidate_acc),
        "promotion_gate": "candidate val_loss lower AND candidate val_accuracy >= baseline - 0.01",
        "promoted": bool(promoted),
        "epochs_completed": len(history.history.get("loss", [])),
        "test_set_used": False
    }

    if promoted:
        new_meta = dict(meta)
        new_meta["model_version"] = candidate_version
        new_meta["model_file"] = candidate_name
        new_meta["created_utc"] = result["created_utc"]
        new_meta["last_retraining"] = result
        # Historical test metrics belong to the original evaluated model.
        new_meta["test_metrics_note"] = (
            "Test metrics in this metadata refer to the original independently "
            "evaluated model; the new version has not been evaluated on the test set."
        )
        tmp = META_PATH.with_suffix(".tmp")
        tmp.write_text(json.dumps(new_meta, indent=2))
        os.replace(tmp, META_PATH)
        result["status"] = "promoted"
    else:
        candidate_path.unlink(missing_ok=True)
        result["status"] = "not_promoted"
        result["reason"] = "Candidate did not pass the validation-only promotion gate."

    REPORTS.mkdir(parents=True, exist_ok=True)
    (REPORTS / "last_retraining.json").write_text(json.dumps(result, indent=2))
    print(json.dumps(result, indent=2))
    return result

if __name__ == "__main__":
    main()
"""
(SRC_DIR / "retrain.py").write_text(textwrap.dedent(retrain_source).strip() + "\n")
(SRC_DIR / "__init__.py").touch(exist_ok=True)
print("Created:", SRC_DIR / "retrain.py")


Created: /content/drive/MyDrive/ecosort/src/retrain.py


## 5. Create the Streamlit dashboard

Dashboard sections:
- API health and API process uptime.
- Active model version, class list and baseline evaluation metrics.
- Visualizations: class distribution, training/validation curves, and test confusion matrix.
- Single-image prediction with uncertainty/manual-review information.
- Bulk labelled-image ZIP upload to the staging area.
- Retraining button that launches `src/retrain.py` and displays whether the candidate was promoted.

The bulk upload format is a ZIP with one folder per class, for example:
`Cardboard/a.jpg`, `Plastic/b.jpg`, `Food Organics/c.jpg`.
Folder names must exactly match the nine class names.

In [5]:
dashboard_source = r"""
import os, sys, json, io, zipfile, time, hashlib, subprocess
from pathlib import Path, PurePosixPath
import requests
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

ROOT = Path(os.getenv("ECOSORT_PROJECT_DIR", Path(__file__).resolve().parent))
sys.path.insert(0, str(ROOT))
from src.predictor import EcoSortPredictor

MODELS = ROOT / "models"
REPORTS = ROOT / "reports"
SPLITS = ROOT / "splits"
INBOX = ROOT / "data_inbox"
META_PATH = MODELS / "model_metadata.json"
METRICS_PATH = REPORTS / "metrics.json"
API_URL = os.getenv("ECOSORT_API_URL", "http://127.0.0.1:8000")

st.set_page_config(page_title="EcoSort AI", page_icon="♻️", layout="wide")
st.title("♻️ EcoSort AI")
st.caption("Waste material classification • model monitoring • controlled retraining")

def read_json(path, default=None):
    if path.exists():
        return json.loads(path.read_text())
    return default if default is not None else {}

if not META_PATH.exists():
    st.error("Model metadata is missing. Run Notebooks 03 and 04.")
    st.stop()
meta = read_json(META_PATH)
metrics = read_json(METRICS_PATH, {})
classes = meta.get("class_names", [])

# API health and uptime
st.subheader("System status")
health = None
try:
    response = requests.get(f"{API_URL}/health", timeout=3)
    response.raise_for_status()
    health = response.json()
    c1, c2, c3 = st.columns(3)
    c1.metric("API status", "Online")
    c2.metric("Model loaded", "Yes" if health.get("model_loaded") else "No")
    uptime = health.get("uptime_seconds")
    c3.metric("API uptime", f"{uptime/3600:.2f} hours" if uptime is not None else "Not reported")
    st.caption(f"API model version: {health.get('model_version', 'unknown')}")
except Exception:
    st.warning(
        f"API not reachable at {API_URL}. Start it in a separate terminal with "
        "`uvicorn src.app:app --host 127.0.0.1 --port 8000`."
    )

st.subheader("Active model")
m1, m2, m3 = st.columns(3)
m1.metric("Model version", str(meta.get("model_version", "unknown")))
m2.metric("Input size", " × ".join(map(str, meta.get("input_size", [224,224,3])[:2])))
m3.metric("Confidence threshold", f"{float(meta.get('confidence_threshold', 0.70)):.0%}")
st.write("**Recognized material classes:**", ", ".join(classes))

# The metrics file records the initial independent evaluation from Notebook 03.
st.subheader("Evaluation snapshot")
test_metrics = metrics.get("test", {})
if test_metrics:
    a, b, c, d = st.columns(4)
    a.metric("Test accuracy (initial model)", f"{float(test_metrics.get('accuracy', 0)):.1%}")
    b.metric("Macro precision", f"{float(test_metrics.get('precision_macro', test_metrics.get('macro_precision', 0))):.1%}")
    c.metric("Macro recall", f"{float(test_metrics.get('recall_macro', test_metrics.get('macro_recall', 0))):.1%}")
    d.metric("Macro F1", f"{float(test_metrics.get('f1_macro', test_metrics.get('macro_f1', 0))):.1%}")
    st.caption(
        "These are the original Notebook 03 held-out test metrics. If a later model is promoted, "
        "these values are historical and are not a new test evaluation."
    )
else:
    st.info("No reports/metrics.json found. Evaluation metrics will appear after Notebook 03.")

# Visualization 1: class distribution in original training data.
st.subheader("Data and model visualizations")
train_csv = SPLITS / "train.csv"
if train_csv.exists():
    train_df = pd.read_csv(train_csv)
    if "label" in train_df and classes:
        counts = train_df["label"].value_counts().sort_index()
        chart_df = pd.DataFrame({
            "Class": [classes[int(i)] for i in counts.index],
            "Training images": counts.values
        }).set_index("Class")
        st.markdown("**1. Training images per class**")
        st.bar_chart(chart_df)
        st.caption("Interpretation: low-count classes have fewer training examples and may be harder to learn.")
else:
    st.info("Train split CSV not found.")

# Visualization 2: training and validation learning curves.
history = metrics.get("history", {})
if history and all(k in history for k in ["loss", "val_loss", "accuracy", "val_accuracy"]):
    st.markdown("**2. Training and validation curves (initial model)**")
    curve_df = pd.DataFrame({
        "Training loss": history["loss"],
        "Validation loss": history["val_loss"],
        "Training accuracy": history["accuracy"],
        "Validation accuracy": history["val_accuracy"]
    })
    st.line_chart(curve_df)
    st.caption("Interpretation: a widening train/validation gap can indicate overfitting; validation trends guide early stopping.")
else:
    st.info("Training history is not available in reports/metrics.json.")

# Visualization 3: confusion matrix.
cm = metrics.get("confusion_matrix_test")
cm_classes = metrics.get("class_names", classes)
if cm:
    st.markdown("**3. Held-out test confusion matrix (initial model)**")
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.imshow(cm, interpolation="nearest")
    ax.set_xticks(range(len(cm_classes)))
    ax.set_yticks(range(len(cm_classes)))
    ax.set_xticklabels(cm_classes, rotation=60, ha="right")
    ax.set_yticklabels(cm_classes)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_title("Confusion matrix")
    fig.tight_layout()
    st.pyplot(fig)
    plt.close(fig)
    st.caption("Interpretation: large off-diagonal values show pairs of classes the model confuses.")
else:
    st.info("Confusion matrix not available.")

# Single image prediction
st.subheader("Try one image")
uploaded_image = st.file_uploader(
    "Upload a waste image", type=["jpg", "jpeg", "png", "bmp"], key="single_image"
)
if uploaded_image is not None:
    try:
        image = Image.open(uploaded_image).convert("RGB")
        st.image(image, caption="Uploaded image", width=320)
        if st.button("Classify image", type="primary"):
            with st.spinner("Loading model and classifying..."):
                predictor = EcoSortPredictor(
                    str(MODELS / meta["model_file"]),
                    str(META_PATH),
                    str(ROOT / "disposal_rules.json")
                )
                result = predictor.predict(io.BytesIO(uploaded_image.getvalue()))
            st.write("**Predicted material:**", result["predicted_class"])
            st.metric("Confidence", f"{result['confidence']:.1%}")
            st.write("**Top predictions:**", result["top_k"])
            if result["manual_review"]:
                st.warning("Manual review required. No disposal stream is recommended.")
            else:
                st.write("**Configured disposal stream:**", result["disposal_stream"])
            st.caption("Disposal mapping is configurable and must be locally validated before real-world use.")
    except Exception as exc:
        st.error(f"Could not process image: {exc}")

# Bulk labelled upload as ZIP
st.subheader("Add labelled images for future training")
st.write(
    "Upload a ZIP archive with one folder per class. Images are staged; they do not immediately "
    "change the active model."
)
zip_file = st.file_uploader("Labelled dataset ZIP", type=["zip"], key="labelled_zip")
if zip_file is not None and st.button("Validate and stage uploaded images"):
    staged, skipped, errors = 0, 0, []
    try:
        archive = zipfile.ZipFile(io.BytesIO(zip_file.getvalue()))
        members = archive.infolist()
        if len(members) > 5000:
            raise ValueError("ZIP contains more than 5,000 entries; split it into smaller uploads.")
        for member in members:
            if member.is_dir():
                continue
            name = member.filename.replace("\\\\", "/")
            rel = PurePosixPath(name)
            if rel.is_absolute() or ".." in rel.parts:
                errors.append(f"Rejected unsafe path: {member.filename}")
                continue
            if len(rel.parts) < 2:
                skipped += 1
                continue
            # Accept a wrapper directory, but use the first recognized class folder.
            class_name = next((part for part in rel.parts[:-1] if part in classes), None)
            if class_name is None:
                skipped += 1
                continue
            suffix = Path(rel.name).suffix.lower()
            if suffix not in {".jpg", ".jpeg", ".png", ".bmp"}:
                skipped += 1
                continue
            raw = archive.read(member)
            if len(raw) > 20 * 1024 * 1024:
                errors.append(f"Rejected oversized image: {member.filename}")
                continue
            try:
                Image.open(io.BytesIO(raw)).verify()
            except Exception:
                errors.append(f"Rejected unreadable image: {member.filename}")
                continue
            digest = hashlib.sha256(raw).hexdigest()[:16]
            target_dir = INBOX / class_name
            target_dir.mkdir(parents=True, exist_ok=True)
            target = target_dir / f"{digest}{suffix}"
            if target.exists():
                skipped += 1
                continue
            target.write_bytes(raw)
            staged += 1
        st.success(f"Staged {staged} images. Skipped {skipped}.")
        if errors:
            st.warning("\\n".join(errors[:20]))
    except Exception as exc:
        st.error(f"Could not read ZIP: {exc}")

st.markdown("**Staged image counts**")
staged_counts = {
    c: sum(1 for p in (INBOX / c).glob("*") if p.is_file())
    if (INBOX / c).exists() else 0 for c in classes
}
st.dataframe(pd.DataFrame(
    [{"Class": k, "Staged images": v} for k, v in staged_counts.items()]
), use_container_width=True)

# Retraining
st.subheader("Retrain and validate a candidate")
st.write(
    "The retraining job starts from the active model. It trains with the original training split "
    "plus staged uploads, then compares the candidate with the active model on validation data only."
)
total_staged = sum(staged_counts.values())
if total_staged == 0:
    st.info("Stage labelled images before starting retraining.")
if st.button("Start retraining", type="primary", disabled=(total_staged == 0)):
    with st.spinner("Retraining can take several minutes. Keep this dashboard open..."):
        env = os.environ.copy()
        env["ECOSORT_PROJECT_DIR"] = str(ROOT)
        process = subprocess.run(
            [sys.executable, str(ROOT / "src" / "retrain.py")],
            cwd=str(ROOT), env=env, capture_output=True, text=True
        )
    if process.stdout:
        st.code(process.stdout[-12000:])
    if process.returncode == 0:
        result_path = REPORTS / "last_retraining.json"
        result = json.loads(result_path.read_text()) if result_path.exists() else {}
        if result.get("promoted"):
            st.success(
                f"Candidate {result.get('candidate_version')} passed validation and was promoted. "
                "Restart the API so it loads the new model."
            )
        else:
            st.warning("Candidate did not pass the promotion gate. The active model was kept.")
        st.json(result)
    else:
        st.error("Retraining failed. The active model was not changed.")
        st.code(process.stderr[-12000:])

last_result = REPORTS / "last_retraining.json"
if last_result.exists():
    st.subheader("Most recent retraining result")
    st.json(read_json(last_result))
"""
(SRC_DIR / "dashboard.py").write_text(textwrap.dedent(dashboard_source).strip() + "\n")
print("Created:", SRC_DIR / "dashboard.py")


Created: /content/drive/MyDrive/ecosort/src/dashboard.py


## 6. Add dashboard requirements and check generated Python files

This cell performs syntax checks only. It does not launch Streamlit, train a model or promote a candidate.

In [6]:
dashboard_requirements = """streamlit
pandas
matplotlib
pillow
requests
"""
req_path = PROJECT_DIR / "requirements-dashboard.txt"
req_path.write_text(dashboard_requirements)

import py_compile
for py_file in [SRC_DIR / "app.py", SRC_DIR / "predictor.py",
                SRC_DIR / "retrain.py", SRC_DIR / "dashboard.py"]:
    py_compile.compile(str(py_file), doraise=True)
    print("Syntax OK:", py_file.relative_to(PROJECT_DIR))

print("\\nCreated files:")
for p in [SRC_DIR / "retrain.py", SRC_DIR / "dashboard.py", req_path]:
    print("-", p.relative_to(PROJECT_DIR), f"({p.stat().st_size:,} bytes)")

Syntax OK: src/app.py
Syntax OK: src/predictor.py
Syntax OK: src/retrain.py
Syntax OK: src/dashboard.py
\nCreated files:
- src/retrain.py (7,712 bytes)
- src/dashboard.py (11,387 bytes)
- requirements-dashboard.txt (44 bytes)


## 7. Verify the staging structure and dashboard inputs

The original test split is intentionally not opened by the dashboard retraining code. This check confirms the train/validation files and model report exist.

In [7]:
import pandas as pd

train_df = pd.read_csv(SPLIT_DIR / "train.csv")
val_df = pd.read_csv(SPLIT_DIR / "val.csv")
assert {"path", "label"}.issubset(train_df.columns)
assert {"path", "label"}.issubset(val_df.columns)
assert len(train_df) > 0 and len(val_df) > 0

if METRICS_PATH.exists():
    metric_data = json.loads(METRICS_PATH.read_text())
    print("Metrics sections:", list(metric_data.keys()))
else:
    print("No metrics.json yet; run Notebook 03's reporting section.")

print("Training rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Classes:", len(class_names))
print("Test split is not used by the retraining script.")

Metrics sections: ['model_version', 'created_utc', 'validation', 'test', 'local_test', 'per_class_test', 'confusion_matrix_test', 'class_names', 'confidence_coverage_test', 'history', 'phase1_epochs']
Training rows: 3325
Validation rows: 716
Classes: 9
Test split is not used by the retraining script.


## 8. Run the application

From the EcoSort project root, open **two terminals**.

**Terminal 1 — API**
```bash
pip install -r requirements-api.txt
uvicorn src.app:app --host 127.0.0.1 --port 8000
```

**Terminal 2 — dashboard**
```bash
pip install -r requirements-dashboard.txt
streamlit run src/dashboard.py
```

Open the local Streamlit URL printed in the terminal (normally `http://localhost:8501`). The dashboard checks the API at `http://127.0.0.1:8000`. If using another host or port, set `ECOSORT_API_URL`.

For Google Colab, the notebook creates the app files in Drive, but the Colab runtime is temporary. For a stable demonstration, clone/download the project and run the API and Streamlit dashboard on your computer or deploy them in the later deployment stage.

## 9. How to prepare the labelled ZIP upload

The ZIP must contain class-named folders. Example:

```text
new_waste_images.zip
├── Cardboard/
│   ├── cardboard_new_01.jpg
│   └── cardboard_new_02.jpg
├── Food Organics/
│   └── food_new_01.jpg
├── Glass/
├── Metal/
├── Miscellaneous Trash/
├── Paper/
├── Plastic/
├── Textile Trash/
└── Vegetation/
```

Images are stored in `data_inbox/<class_name>/`. A folder name is treated as the label, so only upload images whose class has been checked by a person. Do not upload ambiguous or mixed items as if their label were certain.

## 10. What Notebook 05 completes — and what remains

**Implemented by this notebook**
- Streamlit dashboard source code.
- API health display and API process uptime (after restarting the API).
- Model version, confidence threshold and class list.
- Three monitoring visualizations: class counts, training/validation curves and confusion matrix.
- Single-image prediction from the active model.
- Bulk labelled-image ZIP validation and staging.
- Retraining trigger.
- Candidate training using original training data plus staged uploads.
- Validation-only candidate comparison and conditional promotion.

**Still required**
- Run the dashboard and test it in the actual local environment.
- Run a real retraining job with correctly labelled new images and inspect its output.
- Restart the API after a model is promoted.
- Docker packaging, cloud deployment and Locust load testing (next pipeline stage).
- Authentication, upload quotas and background job handling before public production use.

**Evaluation caveat:** the test metrics displayed are from the original Notebook 03 evaluation. A newly promoted model has not been evaluated on the held-out test set. Do not report those historical test metrics as if they belonged to a newly promoted version.

## 11. Troubleshooting notes

- **Dashboard says API offline:** start the FastAPI process in a separate terminal and check `http://127.0.0.1:8000/health`.
- **API uptime is missing:** restart the API after Notebook 05 updates `src/app.py`.
- **No staged images:** check that the ZIP has folders named exactly as the nine model classes.
- **Retraining cannot find RealWaste:** check that the dataset is still under `data_raw/` and contains the `RealWaste` folder.
- **Candidate not promoted:** this is a valid outcome. It means the candidate did not pass the validation gate; the active model remains in place.
- **API still reports the old version after promotion:** restart Uvicorn because Notebook 04 loads the model once when the API process starts.
- **Do not use the test set to force a promotion decision.** Keep it as the final independent evaluation set.